Para conectarnos a SQL Server desede Python usaremos estas 2 librerias:
- pyodbc (trabaja con odbc 64 bits)
- sqlalchemy (se debe combinar con pyodbc)
- pyodbc = front-end y sqlachemy back-end
Para conectarnos a una DB necesitamos una cadena de conexion
Instalar le libreria podbc > pip install pyodbc

import pyodbc
pyodbc.drivers()

In [6]:
import mysql.connector


import os
import mysql.connector

from mysql.connector import Error
from dotenv import load_dotenv


# Cargar variables del archivo .env
load_dotenv()


def create_connection():

    try:

        connection = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )

        if connection.is_connected():
            print("✅ Conexión exitosa a MySQL")
            return connection

    except Error as e:
        print(f"❌ Error conectando a MySQL: {e}")

        return None

#2. Vamos a crear una instancia que contendra los objetos de la BBDD
neptuno = mysql.connector.connect (**cadena_conexion)
#3. Cargar un DataFrame
cursor = neptuno.cursor()
cursor. execute("SELECT * FROM Pedidos")
datos = cursor.fetchall()
pedidos = pd.DataFrame(datos)
#4. Visualizar el dataframe
pedidos

from db.connection import create_connection

def main():

    connection = create_connection()

    if connection:

        cursor = connection.cursor()

        cursor.execute("SELECT DATABASE();")

        result = cursor.fetchone()

        print(f"📦 Base de datos actual: {result[0]}")

        cursor.close()

        connection.close()

        print("🔒 Conexión cerrada")

In [13]:
import pandas as pd

from db.connection import create_connection

In [14]:
neptuno = create_connection()
cursor = neptuno.cursor()
cursor.execute("SELECT * FROM Pedidos")
datos = cursor.fetchall()
pedidos = pd.DataFrame(datos)
pedidos.head()

✅ Conexión correcta a MySQL


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,10248,WELLI,5,1996-07-04,1996-08-01,1996-07-16,3,32.3800,Wilman Kala,Keskuskatu 45,Helsinki,NaN,21240,Finlandia
1,10249,TOMSP,6,1996-07-05,1996-08-16,1996-07-10,1,11.6100,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Alemania
2,10250,HANAR,4,1996-07-08,1996-08-05,1996-07-12,2,65.8300,Hanari Carnes,"Rua do Paço, 67",Río de Janeiro,RJ,05454-876,Brasil
3,10251,VICTE,3,1996-07-08,1996-08-05,1996-07-15,1,41.3400,Victuailles en stock,"2, rue du Commerce",Lyon,NaN,69004,Francia
4,10252,SUPRD,4,1996-07-09,1996-08-06,1996-07-11,2,51.3000,Suprêmes délices,"Boulevard Tirou, 255",Charleroi,NaN,B-6000,Bélgica


Practica 12: Conectarse al servidor de Base de datos SQL Server y traernos de Neptuno 2 conjuntos de datos que cargaremos en 2 dataframes.
- Los pedidos solo de Francia o Alemania
- La tabla entera de detalles de pedidos, con el importe calculado, para las filas cuta cantidad sea mayor de 10.
- Hacer merge de los 2 dataframes y obtener la suma de importe por CiudadDestinatario (que tendrá el alias ciudad)
- Guardar en un fichero de excel

Se recomineda usar el Diseñador de consultas...

In [20]:
# 3. Primer dataframe: pedidos Francia o Alemania
# -----------------------------

query_pedidos = """
SELECT 
    IdPedido,
    CiudadDestinatario AS ciudad,
    PaísDestinatario
FROM Pedidos
WHERE PaísDestinatario = 'Francia'
   OR PaísDestinatario = 'Alemania';
"""

pedidos = pd.read_sql(query_pedidos, neptuno)

# -----------------------------
# 4. Segundo dataframe: detalles con importe calculado
# -----------------------------

query_detalles = """
SELECT
    IdPedido,
    IdProducto,
    PrecioUnidad,
    Cantidad,
    (PrecioUnidad * Cantidad) AS Importe
FROM Detalles_de_pedidos
WHERE Cantidad > 10;
"""

detalles = pd.read_sql(query_detalles, neptuno)

# -----------------------------
# 5. Merge de los dos dataframes
# -----------------------------

resultado = pedidos.merge(
    detalles,
    on="IdPedido",
    how="inner"
)

# -----------------------------
# 6. Suma de importe por ciudad
# -----------------------------

resultado_ciudad = (resultado.groupby("ciudad", as_index=False)["Importe"].sum())

# -----------------------------
# 7. Guardar en Excel
# -----------------------------

ruta_salida = "/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/practica12_resultado.xlsx"

resultado_ciudad.to_excel(ruta_salida, index=False)

resultado_ciudad


/var/folders/6f/lhr87vh51bq38sm9v46389m00000gn/T/ipykernel_84405/3685940057.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pedidos = pd.read_sql(query_pedidos, neptuno)
/var/folders/6f/lhr87vh51bq38sm9v46389m00000gn/T/ipykernel_84405/3685940057.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  detalles = pd.read_sql(query_detalles, neptuno)


,ciudad,Importe
0,Aachen,3634.36
1,Berlín,4395.00
2,Brandenburg,31043.35
3,Cunewalde,116814.99
4,Frankfurt a.M.,20384.47
5,Köln,12292.95
6,Leipzig,4473.80
7,Lille,11282.00
8,Lyon,8990.00
9,Mannheim,1662.00


SQLAlchemy libreria para trabajar con base de datos ideal combinar con pyodbc
Caracteristicas principales:
- Abstraccion de la base de datos (te permite trabajar con objetos de Python en lugar de SQL directamente)
- Puedes definir tablas como clases
- Las consultas se hacen con metodos y expresiones de Python
- Soporta múltiples engine o motores de DDBB:
- SQL Server
- MySQL
- PostgreSQL
- SQLLite
- Oracle
- MariaDB

- Mapea relaciones, gestiona Joins y trabaja con objetos en vez de filas
- Es más potente para grandes volumenes de datos. Soporta lazy loading / eager loading
- Se integra bien con Flask y fastAPI (para monitorizar apis)

In [22]:
# Necesito instalar SQL Alchemy, >pip install sqlalchemy

# 0. Llamar a las librerias
import pandas as pd
from sqlalchemy import create_engine

# 1. Guardar los paramentros de configuracion en variables

# Copiar aqui  todo lo de DAniel


In [27]:
# Version mia para proteccion de datos con Conexion SQL Alchemy

from db.alchemy_connection import get_engine

motor = get_engine()

# Con Pyodbc + sqlalchemy, a parte de select, puedo usar INSERT INTO, UPDATE, DELETE

# Iniciar una transaccion, insertar un registro 
with motor.begin() as conn:
    conn.execute(
        ("""INSERT INTO Clientes (nombre, edad)
         VALUES (:nombre, :edad)
         """),
         {"nombre":"Juan","edad":20}
    )